In [ ]:
import sys
import os




project_path = r"C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder"
if project_path not in sys.path:
    sys.path.append(project_path)

import track_builder as tb


BASE_PATH = r"C:\Users\lamin\Documents\maitrise\ASTD\data"
YEAR      = 2019

MONTHS_TO_LOAD = [1, 2, 3]  # Jan, Feb, Mar

USECOLS   = "default"
SAMPLING  = [0, -1]

COLS_REQUIRED = [
    "shipid",
    "date_time_utc",
    "latitude",
    "longitude",
    "astd_cat",
    "flagname",
    "iceclass",
    "sizegroup_gt"
]


In [ ]:
df_for_track = tb.load_astd_monthly(
        BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
        usecols=USECOLS, sampling=SAMPLING, remove_nan_rows=COLS_REQUIRED
    )


In [ ]:
df = tb.load_astd_monthly(
    BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
    usecols=USECOLS, sampling=None, remove_nan_rows=COLS_REQUIRED
)

In [ ]:
tracks = tb.build_ship_tracks(df_for_track,
                                matching_strategy="balanced",
                                )

In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon
import os

# Create the directory if it doesn't exist (safety check)
os.makedirs(shapefile_dir, exist_ok=True)

# Define the full path
filename = "barents_high_traffic.geojson"
shapefile_path = os.path.join(shapefile_dir, filename)

print(f"Target path: {shapefile_path}")

 

min_lon, max_lon = 15.0, 45.0 
min_lat, max_lat = 68.0, 76.0   

traffic_polygon = Polygon([
    (min_lon, min_lat),
    (max_lon, min_lat),
    (max_lon, max_lat),
    (min_lon, max_lat),
    (min_lon, min_lat)
])

gdf_arctic = gpd.GeoDataFrame(
    {'region': ['arctic_sector']}, 
    geometry=[traffic_polygon], 
    crs="EPSG:4326"
)

try:
    gdf_arctic.to_file(shapefile_path, driver="GeoJSON")
    print(f"Success! Zone saved at: {os.path.abspath(shapefile_path)}")
except Exception as e:
    print(f"Error saving file: {e}")

Target path: C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\data\shapefile\barents_high_traffic.geojson
Success! Zone saved at: C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\data\shapefile\barents_high_traffic.geojson


In [10]:
work = tb.build_light_multi_track_data(track_table=tracks, track_sampling=30, positions_df=df, n_tracks_length=2,
                                        preprocess_positions=True,
                                        bounding_box=shapefile_path)

fig = tb.plot_ship_tracks(
    work,
    color_by="track_id",
    color_mode="categorical",
    show_points=False,
    show_start_end=False,
    map_style="open-street-map",
    title="Tracks with bounding_box",
)
fig.update_layout(showlegend=False)
fig.show()

computing typical speeds...


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:382: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:170: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Cleaning completed: 3628 'ghost' or aberrant points removed.
Loading spatial filter from file: C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\data\shapefile\barents_high_traffic.geojson
  -> Spatial Filter: Keeping 9 tracks out of 30.


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\io\astd_loader.py:647: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

